# HFC deepfake detector -- training (Colab)

One run per session. Set the three variables in **cell 5**, then Run All.

Free Colab reclaims idle sessions and throttles GPU after a few hours per
account, so this notebook checkpoints to Google Drive every 1000 iterations and
resumes automatically. If a session dies, just re-run the whole notebook with
the same three variables.

**Settings > Runtime type > T4 GPU** before you start.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Code and dependencies

In [ ]:
REPO = "https://github.com/rahulbhatyogesh/DIP-Project.git"

import os

if os.path.exists("/content/code"):
    !cd /content/code && git pull -q
else:
    !git clone -q {REPO} /content/code
%cd /content/code

!pip install -q timm albumentations 2>&1 | tail -2
import timm, albumentations
print("timm", timm.__version__, "| albumentations", albumentations.__version__)

In [ ]:
# Architecture check -- should match the paper's Table IV exactly.
from src.models.build import verify_param_counts
assert verify_param_counts(), "parameter counts do not match the paper"

## 2. Face crops

Pulled from the private Kaggle dataset built in `01_preprocess_kaggle.ipynb`.
Needs your API token: kaggle.com > Settings > **Create New Token** downloads
`kaggle.json`. Upload it when prompted (it is not stored in the notebook).

In [ ]:
KAGGLE_DATASET = "rahulbhat05/celebdf-hfc-faces"

from pathlib import Path

if not Path("/content/faces/manifest.csv").exists():
    if not Path.home().joinpath(".kaggle/kaggle.json").exists():
        from google.colab import files
        print("Upload kaggle.json:")
        files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    !pip install -q kaggle
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content --unzip
    !tar -xf /content/faces.tar -C /content

assert Path("/content/faces/manifest.csv").exists(), "crops not found"
!echo "crops: $(find /content/faces -name '*.jpg' | wc -l)"

## 3. Drive, for checkpoints that survive a reclaimed session

Checkpoints only -- the crops stay on local SSD, which is far faster.

In [ ]:
# Pin the version explicitly. Kaggle clients default to the NEWEST version, so
# an empty version created by a stray `datasets version` call will 404 while the
# good data sits in an earlier one. Check the dataset's Data Explorer for which
# version actually carries the bytes.
DATASET = "rahulbhat05/celebdf-hfc-faces/versions/1"

from pathlib import Path
import os

if not Path("/content/faces/manifest.csv").exists():
    !pip install -q kagglehub
    import kagglehub
    src = Path(kagglehub.dataset_download(DATASET))
    print("downloaded to:", src)
    # Kaggle unpacks uploaded tars itself, so this is already the faces/ tree.
    faces = next(src.rglob("manifest.csv")).parent
    if not Path("/content/faces").exists():
        os.symlink(faces, "/content/faces")   # symlink, not a 2.3 GB copy

assert Path("/content/faces/manifest.csv").exists(), "crops not found"
assert Path("/content/faces/splits.json").exists(), "splits.json missing"
# -L because /content/faces is a symlink and find will not descend one otherwise.
!echo "crops: $(find -L /content/faces -name '*.jpg' | wc -l)"

## 4. Pick the run

**This is the only cell that changes between accounts.** The eight Tier 1+2
cells:

| BACKBONE | MODE | PROTOCOL |
|---|---|---|
| `xception` | `none` / `hfc` | `random` / `identity` |
| `efficientnet_b4` | `none` / `hfc` | `random` / `identity` |

Add `MODE = "hfc_only"` for the third row of Table III.

In [ ]:
BACKBONE = "xception"      # xception | efficientnet_b4
MODE     = "hfc"           # none | hfc | hfc_only
PROTOCOL = "identity"      # random | identity

RUN = f"{BACKBONE}_{MODE}_{PROTOCOL}"
OUT = f"{RUNS}/{RUN}"
print("run:", RUN)
print("out:", OUT)

## 5. Train

`--resume` makes this safe to re-run after a disconnect; it picks up at the
last checkpoint. `--max-iters` defaults to the Table IV convergence figure plus
25% headroom, so expect roughly 1-1.5 h for Xception and 2.5-3.5 h for
EfficientNet-B4 on a T4.

Early validation passes look terrible -- BatchNorm running statistics need a few
hundred iterations to settle. Judge the trend, not the first number.

In [ ]:
!python -m src.train \
    --faces-root /content/faces \
    --manifest /content/faces/manifest.csv \
    --splits /content/faces/splits.json \
    --backbone {BACKBONE} --mode {MODE} --protocol {PROTOCOL} \
    --out {OUT} --workers 2 --resume

## 6. Score the test partition

In [ ]:
!mkdir -p {RUNS}/results
!python -m src.eval \
    --ckpt {OUT}/best.pt \
    --faces-root /content/faces \
    --manifest /content/faces/manifest.csv \
    --splits /content/faces/splits.json \
    --part test --out {RUNS}/results/{RUN}.json

In [ ]:
# Training curve, for the report.
import json
import matplotlib.pyplot as plt

log = [json.loads(l) for l in open(f"{OUT}/log.jsonl")]
it = [r["iteration"] for r in log]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(it, [r["train_loss"] for r in log], label="train")
ax[0].plot(it, [r["val"]["loss"] for r in log], label="val")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("BCE loss")
ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_title(RUN)

ax[1].plot(it, [r["val"]["frame"]["auc"] for r in log], label="frame AUC")
if "video" in log[0]["val"]:
    ax[1].plot(it, [r["val"]["video"]["auc"] for r in log], label="video AUC")
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("AUC")
ax[1].legend(); ax[1].set_title("validation")
plt.tight_layout(); plt.savefig(f"{OUT}/curve.png", dpi=150); plt.show()

## 7. Once every run is done

All results land in one Drive folder, so assemble the tables from any session:

```python
from src.figures import build_tables
print(build_tables(f"{RUNS}/results"))
```

and the paper's Fig. 1 reproduction from any HFC checkpoint:

```bash
!python -m src.figures activations --ckpt {OUT}/best.pt \
    --faces-root /content/faces --manifest /content/faces/manifest.csv \
    --out {RUNS}/figures
```